In [ ]:
import os
import pandas as pd
from glob import glob

from util import calc_perceptual_metrics


attack_dir = 'exp/attack'
percept_dir = 'exp/percept'

In [ ]:
res = []
for attack_name in os.listdir(attack_dir):
    ben_audio_files = sorted(glob(os.path.join(attack_dir, attack_name, '*.wav')))
    param_names = list(filter(lambda x: '.log' not in x and x.startswith('alpha') and 'len15' in x, os.listdir(os.path.join(attack_dir, attack_name))))
    for param_name in param_names:
        sub_param_names = glob(os.path.join(attack_dir, attack_name, param_name, "*.jsonl"))
        for sub_param_name in sub_param_names:
            carrier_type = os.path.basename(sub_param_name).split('_carrier')[0].split('_')[-1]
            adv_audio_files = sorted(glob(os.path.join(attack_dir, attack_name, param_name, 'wav', '*.wav')))
            if attack_name in ['caa_linf', 'caa_l2'] or carrier_type != 'speech':
                adv_audio_files = list(filter(lambda x: carrier_type in x, adv_audio_files))
            for adv_audio_file in adv_audio_files:
                if attack_name == 'caa_linf' or attack_name == 'caa_l2':
                    ben_audio_file = '_'.join(os.path.basename(adv_audio_file).split('_')[1:3]) + '_len_15_additive_ben.wav'
                    ben_audio_file = os.path.join(attack_dir, attack_name, ben_audio_file)
                else:
                    if carrier_type == 'speech':
                        ben_audio_file = os.path.join(attack_dir, attack_name, 'speech_carrier0_len_15_rir_ln_3200_ota.wav')
                    else:
                        ben_audio_file = '_'.join(os.path.basename(adv_audio_file).split('_')[1:3]) + '_len_15_rir_ln_3200_ota.wav'
                        ben_audio_file = os.path.join(attack_dir, attack_name, ben_audio_file)
                snr, mcd, stoi, pesq = calc_perceptual_metrics(ben_audio_file, adv_audio_file)
                res.append([
                    '_'.join(attack_name.split('_')[:-1]),
                    attack_name.split('_')[-1], 
                    carrier_type,
                    param_name,
                    os.path.basename(ben_audio_file), 
                    os.path.basename(adv_audio_file), 
                    snr, mcd, stoi, pesq
                ])
df = pd.DataFrame(res, columns=['lalm', 'attack', 'carrier_type', 'param', 'ben_audio_file', 'adv_audio_file', 'snr', 'mcd', 'stoi', 'pesq'])
df.to_csv(os.path.join(percept_dir, 'percept_result.csv'))

In [ ]:
df = pd.read_csv(os.path.join(percept_dir, 'percept_result.csv'))
l2_df = df[df['attack'] == 'caa_l2']
linf_df = df[df['attack'] == 'caa_linf']
cov_df = df[df['attack'] == 'caa']
print(linf_df.groupby(['carrier_type'])[['snr', 'mcd', 'stoi', 'pesq']].agg(['mean']))
print(l2_df.groupby(['carrier_type'])[['snr', 'mcd', 'stoi', 'pesq']].agg(['mean']))
print(cov_df.groupby(['carrier_type'])[['snr', 'mcd', 'stoi', 'pesq']].agg(['mean']))

                    snr       mcd      stoi      pesq
                   mean      mean      mean      mean
carrier_type                                         
music         14.711124  5.707446  0.844712  1.704326
sound         12.771240  4.242735  0.702921  2.171385
speech         9.879060  8.365056  0.774843  1.185106
                    snr       mcd      stoi      pesq
                   mean      mean      mean      mean
carrier_type                                         
music         27.164809  3.604287  0.956667  2.983495
sound         25.306475  1.961117  0.920598  3.548826
speech        22.154893  5.561034  0.906434  2.230956
                    snr       mcd      stoi      pesq
                   mean      mean      mean      mean
carrier_type                                         
music         30.057206  2.804454  0.920116  3.562729
sound         28.610564  2.371465  0.880632  3.202537
speech        29.268930  4.169168  0.910687  3.063922
